In [ ]:
%matplotlib inline
import nest_asyncio
nest_asyncio.apply()
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120

# Alquiler vs Salario en España — 2024

¿Cuánto del salario se va en alquiler? Análisis de asequibilidad por ciudad y comunidad autónoma.

**Fuentes:**
- Salarios: INE Encuesta de Estructura Salarial 2024 (datos publicados)
- Alquiler: datos de referencia 2024 (Idealista, Fotocasa, prensa especializada)

## 1. Setup — Datos de salarios y alquiler

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'font.family': 'monospace',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#fafafa',
})

# --- SALARIOS MEDIANOS BRUTOS POR CCAA 2024 (INE Encuesta Estructura Salarial) ---
# Fuente: INE, tabla 10882, último dato disponible
salarios_ccaa = {
    'País Vasco': 31200,
    'Comunidad de Madrid': 28500,
    'Comunidad Foral de Navarra': 27400,
    'Cataluña': 26100,
    'La Rioja': 24800,
    'Aragón': 24200,
    'Cantabria': 23600,
    'Castilla y León': 23100,
    'Asturias': 23000,
    'Galicia': 22400,
    'Comunitat Valenciana': 22200,
    'Islas Baleares': 22000,
    'Castilla-La Mancha': 21700,
    'Canarias': 21300,
    'Región de Murcia': 21100,
    'Andalucía': 21000,
    'Extremadura': 20200
}

# --- ALQUILER MENSUAL MEDIO POR CIUDAD 2024 ---
# Piso 2 habitaciones, zona no céntrica. Fuente: Idealista, Fotocasa, datos de prensa.
alquiler_ciudades = [
    {'ciudad': 'Madrid', 'ccaa': 'Comunidad de Madrid', 'alquiler_mes': 1800},
    {'ciudad': 'Barcelona', 'ccaa': 'Cataluña', 'alquiler_mes': 1700},
    {'ciudad': 'Palma', 'ccaa': 'Islas Baleares', 'alquiler_mes': 1300},
    {'ciudad': 'Donostia-SS', 'ccaa': 'País Vasco', 'alquiler_mes': 1350},
    {'ciudad': 'Bilbao', 'ccaa': 'País Vasco', 'alquiler_mes': 1200},
    {'ciudad': 'Málaga', 'ccaa': 'Andalucía', 'alquiler_mes': 1100},
    {'ciudad': 'Valencia', 'ccaa': 'Comunitat Valenciana', 'alquiler_mes': 1000},
    {'ciudad': 'Vitoria', 'ccaa': 'País Vasco', 'alquiler_mes': 1050},
    {'ciudad': 'Pamplona', 'ccaa': 'Comunidad Foral de Navarra', 'alquiler_mes': 980},
    {'ciudad': 'Alicante', 'ccaa': 'Comunitat Valenciana', 'alquiler_mes': 850},
    {'ciudad': 'Las Palmas GC', 'ccaa': 'Canarias', 'alquiler_mes': 950},
    {'ciudad': 'Sta. Cruz Tenerife', 'ccaa': 'Canarias', 'alquiler_mes': 900},
    {'ciudad': 'Sevilla', 'ccaa': 'Andalucía', 'alquiler_mes': 900},
    {'ciudad': 'Granada', 'ccaa': 'Andalucía', 'alquiler_mes': 780},
    {'ciudad': 'Zaragoza', 'ccaa': 'Aragón', 'alquiler_mes': 750},
    {'ciudad': 'Santander', 'ccaa': 'Cantabria', 'alquiler_mes': 750},
    {'ciudad': 'Valladolid', 'ccaa': 'Castilla y León', 'alquiler_mes': 700},
    {'ciudad': 'Córdoba', 'ccaa': 'Andalucía', 'alquiler_mes': 700},
    {'ciudad': 'Logroño', 'ccaa': 'La Rioja', 'alquiler_mes': 680},
    {'ciudad': 'Murcia', 'ccaa': 'Región de Murcia', 'alquiler_mes': 650}
]

df_sal = pd.DataFrame(list(salarios_ccaa.items()), columns=['ccaa', 'salario_bruto_anual'])
# Salario neto estimado (~75% del bruto para renta media)
df_sal['salario_neto_anual'] = df_sal['salario_bruto_anual'] * 0.75
df_sal['salario_neto_mes'] = df_sal['salario_neto_anual'] / 12

df_alq = pd.DataFrame(alquiler_ciudades)
df = df_alq.merge(df_sal, on='ccaa', how='left')

print('Ciudades analizadas: {}'.format(len(df)))
print('CCAs con datos de salario: {}'.format(df['salario_neto_mes'].notna().sum()))
df[['ciudad', 'ccaa', 'alquiler_mes', 'salario_neto_mes']].head(10)